In [12]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly_express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import yfinance as yf
import pandas_market_calendars as mcal

In [13]:
ruta_datos = os.path.join("..", "DATOS")
ruta_datos_original = os.path.join(ruta_datos, 'ORIGINALES')
ruta_datos_transformados = os.path.join(ruta_datos, 'TRANSFORMADOS')
ruta_datos_modelos = os.path.join(ruta_datos, 'MODELOS')
ruta_datos_graficos = os.path.join(ruta_datos, 'GRAFICOS')

if not os.path.exists(ruta_datos):
    os.makedirs(ruta_datos) 
    print(f"Carpeta '{ruta_datos}' creada con éxito.")
else:
    print(f"La carpeta '{ruta_datos}' ya existe.")

if not os.path.exists(ruta_datos_original):
    os.makedirs(ruta_datos_original) 
    print(f"Carpeta '{ruta_datos_original}' creada con éxito.")
else:
    print(f"La carpeta '{ruta_datos_original}' ya existe.")

if not os.path.exists(ruta_datos_transformados):
    os.makedirs(ruta_datos_transformados) 
    print(f"Carpeta '{ruta_datos_transformados}' creada con éxito.")
else:
    print(f"La carpeta '{ruta_datos_transformados}' ya existe.")

if not os.path.exists(ruta_datos_modelos):
    os.makedirs(ruta_datos_modelos) 
    print(f"Carpeta '{ruta_datos_modelos}' creada con éxito.")
else:
    print(f"La carpeta '{ruta_datos_modelos}' ya existe.")

if not os.path.exists(ruta_datos_graficos):
    os.makedirs(ruta_datos_graficos) 
    print(f"Carpeta '{ruta_datos_graficos}' creada con éxito.")
else:
    print(f"La carpeta '{ruta_datos_graficos}' ya existe.")

La carpeta '..\DATOS' ya existe.
La carpeta '..\DATOS\ORIGINALES' ya existe.
La carpeta '..\DATOS\TRANSFORMADOS' ya existe.
La carpeta '..\DATOS\MODELOS' ya existe.
La carpeta '..\DATOS\GRAFICOS' ya existe.


In [14]:
dir_origen = '../DATOS/ORIGINALES'

dfs = [
    pd.read_csv(os.path.join(r, f))
    for r, _, files in os.walk(dir_origen)
    for f in files if f.lower().endswith('_marketstack.csv')
]

df = pd.concat(dfs, ignore_index=True)
df.to_csv("../DATOS/TRANSFORMADOS/dow_jones_consolidado.csv", index=False)

ValueError: No objects to concatenate

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df["date"] = pd.to_datetime(df["date"])
if df["date"].dt.tz is None:
    df["date"] = df["date"].dt.tz_localize("UTC")

df["date"] = df["date"].dt.tz_convert("America/New_York")

nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(
    start_date=df["date"].min().date(), end_date=df["date"].max().date()
)
dias_habiles_nyse = schedule.index.date

es_dia_laborable = df["date"].dt.date.isin(dias_habiles_nyse)

hora_apertura = pd.to_datetime("09:30").time()
hora_cierre = pd.to_datetime("16:00").time()

es_horario_mercado = (df["date"].dt.time >= hora_apertura) & (
    df["date"].dt.time <= hora_cierre
)

df_horario_mercado = df[es_dia_laborable & es_horario_mercado].copy()

df_horario_mercado["date"] = df_horario_mercado["date"].dt.tz_localize(None)
df_horario_mercado

,open,high,low,last,close,volume,date,symbol,exchange
3,213.133,213.730,212.985,NaN,213.480,NaN,2018-08-16 16:00:00,AAPL,IEXG
4,213.273,213.300,213.133,NaN,213.133,NaN,2018-08-16 15:30:00,AAPL,IEXG
5,212.785,213.465,212.785,NaN,213.133,NaN,2018-08-16 15:00:00,AAPL,IEXG
6,212.850,213.198,212.785,NaN,212.785,NaN,2018-08-16 14:30:00,AAPL,IEXG
7,211.780,213.198,211.780,NaN,212.785,NaN,2018-08-16 14:00:00,AAPL,IEXG
...,...,...,...,...,...,...,...,...,...
2744413,234.500,235.600,234.170,235.300,234.090,291094.0,2021-06-29 13:07:00,V,IEXG
2744414,234.500,235.600,234.170,235.235,234.090,290571.0,2021-06-29 13:05:00,V,IEXG
2744415,234.500,235.600,234.170,235.270,234.090,290254.0,2021-06-29 13:04:00,V,IEXG
2744416,234.500,235.600,234.170,235.255,234.090,289854.0,2021-06-29 13:03:00,V,IEXG


In [ ]:
missings = df_horario_mercado.isna().sum()
print("Missings por columna:", missings)

Missings por columna: open             0
high             0
low              0
last        155101
close        78043
volume      152585
date             0
symbol           0
exchange         0
dtype: int64


In [ ]:
df_horario_mercado["fecha_dia"] = df_horario_mercado["date"].dt.date
dias_reales = df_horario_mercado.groupby("symbol")["fecha_dia"].unique()

dias_faltantes = [
    {"symbol": activo, "fecha_faltante": dia.strftime("%Y-%m-%d")}
    for activo, fechas in dias_reales.items()
    for dia in set(pd.bdate_range(min(fechas), max(fechas)).date) - set(fechas)
]

df_dias_faltantes = pd.DataFrame(dias_faltantes).sort_values(
    ["symbol", "fecha_faltante"]
)
df_dias_faltantes

,symbol,fecha_faltante
14,AAPL,2018-07-04
0,AAPL,2018-09-03
7,AAPL,2018-11-22
4,AAPL,2018-12-05
11,AAPL,2018-12-25
...,...,...
383,V,2021-01-01
385,V,2021-01-18
395,V,2021-02-15
407,V,2021-04-02


In [ ]:
print(df_horario_mercado.groupby("symbol")["date"].agg(["min", "max"]))

                       min                 max
symbol                                        
AAPL   2018-06-06 13:30:00 2021-06-30 16:00:00
AMGN   2020-05-29 11:35:00 2021-06-30 16:00:00
AXP    2020-05-29 11:30:59 2021-06-30 16:00:00
BA     2020-05-29 11:35:51 2021-06-30 16:00:00
CAT    2018-06-11 13:30:00 2021-06-30 16:00:00
CRM    2020-05-29 11:31:30 2021-06-30 16:00:00
CSCO   2020-05-29 11:31:26 2021-06-30 16:00:00
CVX    2020-05-29 11:31:23 2021-06-30 16:00:00
DIS    2020-05-29 11:35:00 2021-06-30 16:00:00
DOW    2020-05-29 11:35:56 2021-06-30 16:00:00
GS     2018-06-11 13:30:00 2021-06-30 16:00:00
HD     2020-05-29 13:55:51 2021-06-30 16:00:00
IBM    2020-05-29 13:56:29 2021-06-30 16:00:00
INTC   2018-06-07 14:00:00 2021-06-30 16:00:00
JNJ    2020-05-29 13:56:17 2021-06-30 16:00:00
JPM    2020-05-29 13:56:13 2021-06-30 16:00:00
KO     2020-05-29 13:56:55 2021-06-30 16:00:00
MCD    2020-05-29 13:54:54 2021-06-30 16:00:00
MMM    2020-05-29 13:54:34 2021-06-30 16:00:00
MRK    2020-0

In [ ]:
jornadas = (
    df_horario_mercado.groupby(["symbol", df_horario_mercado["date"].dt.date])
    .agg(Apertura=("date", "min"), Cierre=("date", "max"))
    .reset_index()
)

df_jornadas_irregulares = jornadas[
    (jornadas["Apertura"].dt.strftime("%H:%M") != "09:30")
    | (jornadas["Cierre"].dt.strftime("%H:%M") != "16:00")
].copy()

df_jornadas_irregulares["Fecha"] = df_jornadas_irregulares["Apertura"].dt.strftime("%Y-%m-%d")
df_jornadas_irregulares["Duracion_Horas"] = (
    (df_jornadas_irregulares["Cierre"] - df_jornadas_irregulares["Apertura"]).dt.total_seconds() / 3600
).round(2)
df_jornadas_irregulares["Apertura"] = df_jornadas_irregulares["Apertura"].dt.strftime("%H:%M")
df_jornadas_irregulares["Cierre"] = df_jornadas_irregulares["Cierre"].dt.strftime("%H:%M")

df_jornadas_irregulares = df_jornadas_irregulares[
    ["symbol", "Fecha", "Apertura", "Cierre", "Duracion_Horas"]
]
df_jornadas_irregulares

,symbol,Fecha,Apertura,Cierre,Duracion_Horas
0,AAPL,2018-06-06,13:30,16:00,2.50
1,AAPL,2018-06-07,13:30,16:00,2.50
2,AAPL,2018-06-08,13:30,16:00,2.50
3,AAPL,2018-06-11,13:30,16:00,2.50
4,AAPL,2018-06-12,13:30,16:00,2.50
...,...,...,...,...,...
10033,V,2021-05-27,09:31,16:00,6.48
10034,V,2021-05-28,09:31,16:00,6.48
10035,V,2021-06-01,09:31,16:00,6.48
10039,V,2021-06-07,09:31,16:00,6.48


In [ ]:
df_horario_mercado["gap"] = df_horario_mercado.groupby(
    ["symbol", df_horario_mercado["date"].dt.date]
)["date"].diff()

res = df_horario_mercado.loc[
    df_horario_mercado.groupby("symbol")["gap"].idxmax()
].copy()

res["Desde_Hora"] = (res["date"] - res["gap"]).dt.strftime("%H:%M:%S")
res["Hasta_Hora"] = res["date"].dt.strftime("%H:%M:%S")
res["Fecha"] = res["date"].dt.strftime("%Y-%m-%d")
res["Minutos"] = (res["gap"].dt.total_seconds() / 60).round(1)
print("\nMAYOR SALTO INTRADÍA POR ACTIVO (DESDE / HASTA)")
print(
    res[["symbol", "Fecha", "Desde_Hora", "Hasta_Hora", "Minutos"]]
    .sort_values("Minutos", ascending=False)
    .to_string(index=False)
)


MAYOR SALTO INTRADÍA POR ACTIVO (DESDE / HASTA)
symbol      Fecha Desde_Hora Hasta_Hora  Minutos
  AAPL 2020-09-21   09:30:00   16:00:00    390.0
  AMGN 2021-03-05   09:30:00   16:00:00    390.0
   AXP 2020-10-23   09:30:00   16:00:00    390.0
    BA 2020-10-27   09:30:00   16:00:00    390.0
   CAT 2020-10-21   09:30:00   16:00:00    390.0
   CRM 2020-10-13   09:30:00   16:00:00    390.0
  CSCO 2020-06-16   09:30:00   16:00:00    390.0
   CVX 2020-10-22   09:30:00   16:00:00    390.0
   DIS 2020-10-23   09:30:00   16:00:00    390.0
   DOW 2020-10-22   09:30:00   16:00:00    390.0
    GS 2020-10-22   09:30:00   16:00:00    390.0
    HD 2020-10-22   09:30:00   16:00:00    390.0
   IBM 2020-10-23   09:30:00   16:00:00    390.0
  INTC 2020-10-26   09:30:00   16:00:00    390.0
   JNJ 2020-10-23   09:30:00   16:00:00    390.0
   JPM 2021-03-15   09:30:00   16:00:00    390.0
    KO 2021-03-03   09:30:00   16:00:00    390.0
   MCD 2020-06-16   09:30:00   16:00:00    390.0
   MMM 2020-10-22   

In [ ]:
df_nulos_close = (
    df_horario_mercado.groupby("symbol")["close"]
    .apply(lambda x: (x.isnull().sum() / len(x)) * 100)
    .round(2)
    .reset_index(name="pct_nulos_close")
    .sort_values("pct_nulos_close", ascending=False)
)

print(df_nulos_close.to_string(index=False))

symbol  pct_nulos_close
   TRV             6.39
  AMGN             4.89
   AXP             4.86
    BA             4.27
   MMM             4.21
   DOW             4.21
  CSCO             4.09
   CRM             3.99
   CAT             3.98
  MSFT             3.97
   IBM             3.94
   CVX             3.89
   DIS             3.86
   UNH             3.83
   JPM             3.65
   MCD             3.64
    HD             3.61
   JNJ             3.55
    GS             3.51
     V             3.49
  INTC             3.44
    KO             3.30
    PG             3.30
   MRK             3.26
  AAPL             3.08
   NKE             3.00


In [ ]:
if not df_dias_faltantes.empty:
    tickers = df_dias_faltantes["symbol"].unique().tolist()
    f_min = df_dias_faltantes["fecha_faltante"].min()
    f_max = (
        pd.to_datetime(df_dias_faltantes["fecha_faltante"].max())
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")

    df_yf_dias = (
        yf.download(
            tickers, start=f_min, end=f_max, interval="1d", group_by="ticker"
        )
        .stack(level=0, future_stack=True)
        .reset_index()
    )
    df_yf_dias = df_yf_dias.rename(
        columns={
            "Date": "date",
            "Ticker": "symbol",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )
    df_yf_dias["date"] = pd.to_datetime(df_yf_dias["date"])

    df_dias_rec = pd.merge(
        df_dias_faltantes,
        df_yf_dias,
        left_on=["symbol", "fecha_faltante"],
        right_on=["symbol", df_yf_dias["date"].dt.strftime("%Y-%m-%d")],
    ).drop(columns=["fecha_faltante"])
else:
    df_dias_rec = pd.DataFrame()

cols = ["symbol", "date", "open", "high", "low", "close", "volume"]
df_consolidado_completo = (
    pd.concat([df_horario_mercado[cols], df_dias_rec[cols]], ignore_index=True)
    .drop_duplicates(subset=["symbol", "date"])
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)

[*********************100%***********************]  26 of 26 completed


In [ ]:
df_consolidado_completo["date"] = pd.to_datetime(
    df_consolidado_completo["date"]
).dt.tz_localize(None)
df_consolidado_completo["fecha_dia"] = df_consolidado_completo["date"].dt.date

nyse = mcal.get_calendar("NYSE")
faltantes = [
    {"symbol": symbol, "fecha_faltante": dia.strftime("%Y-%m-%d")}
    for symbol, g in df_consolidado_completo.groupby("symbol")
    for dia in set(
        nyse.schedule(
            start_date=g["fecha_dia"].min(), end_date=g["fecha_dia"].max()
        ).index.date
    )
    - set(g["fecha_dia"].unique())
]
df_dias_faltantes_post = pd.DataFrame(faltantes)

jornadas = (
    df_consolidado_completo.groupby(["symbol", "fecha_dia"])
    .agg(
        apertura=("date", lambda x: x.min().strftime("%H:%M")),
        cierre=("date", lambda x: x.max().strftime("%H:%M")),
    )
    .reset_index()
)

df_jornadas_irregulares_post = jornadas[
    (jornadas["apertura"] != "09:30") | (jornadas["cierre"] != "16:00")
].reset_index(drop=True)

print(f"Días faltantes restantes: {len(df_dias_faltantes_post)}")
print(f"Jornadas irregulares restantes: {len(df_jornadas_irregulares_post)}")

Días faltantes restantes: 0
Jornadas irregulares restantes: 8371


In [ ]:
df_pivot = df_consolidado_completo.pivot(index='date', columns='symbol').swaplevel(axis=1).sort_index(axis=1)
columnas_validas = ["open", "high", "low", "close", "volume"]
df_pivot = df_pivot.loc[:, (slice(None), columnas_validas)]

In [ ]:
agg_dict = {}
for col in df_pivot.columns:
    ticker, metric = col
    
    if metric == 'open':
        agg_dict[col] = 'first' 
    elif metric == 'high':
        agg_dict[col] = 'max'  
    elif metric == 'low':
        agg_dict[col] = 'min'  
    elif metric == 'close':
        agg_dict[col] = 'last' 
    elif metric == 'volume':
        agg_dict[col] = 'sum'  

df_daily = df_pivot.groupby(df_pivot.index.normalize()).agg(agg_dict)

df_daily = df_daily.dropna(how='all')

In [ ]:
tickers = df['symbol'].unique().tolist()

datos_yf = yf.download(tickers, start="2018-06-06", end="2021-06-30", auto_adjust=False, actions=True)

splits_data = datos_yf['Stock Splits']

tickers_with_splits = splits_data.loc[:, (splits_data > 0).any(axis=0)]

actual_splits = tickers_with_splits[(tickers_with_splits > 0).any(axis=1)]

print(actual_splits.dropna(how='all'))

[*********************100%***********************]  26 of 26 completed

Ticker      AAPL    MRK
Date                   
2020-08-31   4.0  0.000
2021-06-03   0.0  1.048


In [ ]:
prices_to_adjust = ['open', 'high', 'low', 'close']

for split_date, row in actual_splits.iterrows():
    split_date_str = split_date.strftime('%Y-%m-%d')
    
    for ticker, ratio in row.items():
        if ratio > 0.0:
            print(f"Aplicando el ajuste de split {ratio}-for-1 para {ticker} previo previo a {split_date_str}")
            
            pre_split_mask = df_daily.index.normalize() <= split_date_str
            
            for col in prices_to_adjust:
                df_daily.loc[pre_split_mask, (ticker, col)] /= ratio
            
            df_daily.loc[pre_split_mask, (ticker, 'volume')] *= ratio

print("\nTodos los ajustes de split aplicados.")

Aplicando el ajuste de split 4.0-for-1 para AAPL previo previo a 2020-08-31
Aplicando el ajuste de split 1.048-for-1 para MRK previo previo a 2021-06-03

Todos los ajustes de split aplicados.


In [ ]:
f_ini, f_fin = "2018-06-06", "2025-06-06"
nyse = mcal.get_calendar("NYSE")

dias_nyse = nyse.schedule(start_date=f_ini, end_date=f_fin).index.tz_localize(None)

df_daily.index = pd.to_datetime(df_daily.index).tz_localize(None)
df_daily = df_daily.reindex(dias_nyse)
df_daily.columns.names = ['Ticker', 'Atributo']

tickers_con_huecos = []
for tk in df_daily.columns.get_level_values(0).unique():
    if df_daily[(tk, 'close')].isna().any():
        tickers_con_huecos.append(tk)

if tickers_con_huecos:
    print(f"Descargando datos históricos completos para {len(tickers_con_huecos)} tickers...")
    
    new_data = yf.download(tickers_con_huecos, start=f_ini, end=f_fin, interval="1d", auto_adjust=False)
    
    if not new_data.empty:
        if isinstance(new_data.columns, pd.MultiIndex):
            new_data.columns = new_data.columns.swaplevel(0, 1)
        else:
            new_data.columns = pd.MultiIndex.from_product([[tickers_con_huecos[0]], new_data.columns])
            
        new_data.columns = new_data.columns.set_names(['Ticker', 'Atributo'])
        new_data = new_data.rename(columns=str.lower, level=1)
        
        new_data.index = pd.to_datetime(new_data.index).tz_localize(None)
        new_data = new_data.reindex(dias_nyse)
        
        df_daily = df_daily.combine_first(new_data)

print("\nRangos finales en el DataFrame Consolidado")
for tk in df_daily.columns.get_level_values(0).unique():
    fechas = df_daily[tk].dropna(how='all').index
    if not fechas.empty:
        print(f"{tk}: Min {fechas.min().strftime('%Y-%m-%d')} | Max {fechas.max().strftime('%Y-%m-%d')} | NaNs Restantes: {df_daily[(tk, 'close')].isna().sum()}")

Descargando datos históricos completos para 26 tickers...


[*********************100%***********************]  26 of 26 completed



Rangos finales en el DataFrame Consolidado
AAPL: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
AMGN: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
AXP: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
BA: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
CAT: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
CRM: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
CSCO: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
CVX: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
DIS: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
DOW: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 198
GS: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
HD: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
IBM: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
INTC: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
JNJ: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
JPM: Min 2018-06-06 | Max 2025-06-05 | NaNs Restantes: 1
KO: Min 2018-06-06 | Max 2025-06-05 | NaN

In [ ]:
tickers = df_daily.columns.get_level_values(0).unique().tolist()
tickers = [str(t).strip() for t in tickers] 

datos_yf = yf.download(tickers, start="2018-06-06", end="2025-06-06", auto_adjust=False, actions=True, progress=False)

div_data = datos_yf['Dividends']
close_data = datos_yf['Close']

actual_divs = div_data.loc[:, (div_data > 0).any(axis=0)]
actual_divs = actual_divs[(actual_divs > 0).any(axis=1)]

prices_to_adjust = ['open', 'high', 'low', 'close']

for div_date, row in actual_divs.iterrows():
    div_date_str = div_date.strftime('%Y-%m-%d')
    for ticker, div_amount in row.items():
        if pd.notna(div_amount) and div_amount > 0.0:
            
            prior_closes = close_data[ticker].loc[close_data.index < div_date_str]
            
            if not prior_closes.empty:
                prev_close = prior_closes.iloc[-1]
                if pd.notna(prev_close) and prev_close > 0:
                    adj_ratio = 1 - (div_amount / prev_close)
                    
                    pre_div_mask = df_daily.index.normalize() < div_date_str
                    for col in prices_to_adjust:
                        df_daily.loc[pre_div_mask, (ticker, col)] *= adj_ratio

In [ ]:
window = 5
threshold = 0.15 

for tk in df_daily.columns.get_level_values(0).unique():
    if (tk, 'close') in df_daily.columns:
        close_series = df_daily[tk]['close']
        
        rolling_med = close_series.rolling(window=window, center=True).median()
        
        deviation = np.abs((close_series - rolling_med) / rolling_med)
        
        df_daily.loc[deviation > threshold, (tk, 'close')] = np.nan
        df_daily[(tk, 'close')] = df_daily[(tk, 'close')].interpolate(method='linear')

In [ ]:
df_daily = df_daily.ffill()

df_daily = df_daily.dropna(how='all')

remaining_nas = df_daily.isna().sum()
print("NAs restantes por columna:\n", remaining_nas[remaining_nas > 0])

NAs restantes por columna:
 Ticker  Atributo 
DOW     open         197
        high         197
        low          197
        close        197
        adj close    197
dtype: int64


In [ ]:
raw_tickers = df_daily.columns.get_level_values(0).unique()
tickers_to_plot = [tk for tk in raw_tickers if isinstance(tk, str) and tk.isupper() and 1 <= len(tk) <= 5]

fig = make_subplots(
    rows=len(tickers_to_plot), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=tickers_to_plot
)

for i, tk in enumerate(tickers_to_plot, start=1):
    ticker_close = df_daily[tk]['close'].dropna()
    
    fig.add_trace(
        go.Scatter(
            x=ticker_close.index, 
            y=ticker_close.values, 
            mode='lines', 
            name=tk,
            line=dict(width=1.5)
        ),
        row=i, 
        col=1
    )

fig.update_layout(
    height=300 * len(tickers_to_plot), 
    title_text="Precios Históricos en Close",
    template="plotly_white",
    showlegend=False
)

fig.show()

In [ ]:
df_daily.to_parquet("../DATOS/TRANSFORMADOS/df_diario.parquet")